### The Quadratic Assignment Problem

Assign $N$ factories to $N$ cities to minimize total communication costs.

**Parameters:**
* $N$: total number of factories and cities
* $t_{ik}$: communication frequency between factory $i$ and factory $k$
* $d_{jl}$: distance between city $j$ and city $l$
* $c_{ijkl}$: cost coefficient, where $c_{ijkl} = t_{ik}d_{jl}$

#### Mathematical Model

**Decision Variables:**
$x_{ij} = 1$ if factory $i$ is assigned to city $j$, $0$ otherwise:

$$x_{ij} \in \{0, 1\} \quad i=1,\dots,N; \ j=1,\dots,N$$

**Objective:**
Minimize total communication costs:

$$\text{minimize } z = \sum_{i=1}^{N} \sum_{j=1}^{N} \sum_{k=1}^{N} \sum_{l=1}^{N} c_{ijkl} x_{ij} x_{kl}$$ 

**Constraints:**
1. Factory Assignment: Each factory must be assigned to exactly one city.

$$\sum_{j=1}^{N} x_{ij} = 1 \quad i = 1, \dots, N$$ 

2. City Assignment: Each city must have exactly one factory assigned to it.

$$\sum_{i=1}^{N} x_{ij} = 1 \quad j = 1, \dots, N$$

In [1]:
import numpy as np
import scipy.sparse as sp

import cplex as cp

In [2]:
def mixed_integer_quadratic_programming(direction, A, senses, b, c, Q, l, u, types, names):
    # create an empty optimization problem
    prob = cp.Cplex()

    # add decision variables to the problem including their linear coefficients in objective and ranges
    prob.variables.add(obj = c.tolist(), lb = l.tolist(), ub = u.tolist(), types = types.tolist(), names = names.tolist())
    
    # add quadratic coefficients in objective
    row_indices, col_indices = Q.nonzero()
    prob.objective.set_quadratic_coefficients(zip(row_indices.tolist(), col_indices.tolist(), Q.data.tolist()))

    # define problem type
    if direction == "maximize":
        prob.objective.set_sense(prob.objective.sense.maximize)
    else:
        prob.objective.set_sense(prob.objective.sense.minimize)

    # add constraints to the problem including their directions and right-hand side values
    prob.linear_constraints.add(senses = senses.tolist(), rhs = b.tolist())

    # add coefficients for each constraint
    row_indices, col_indices = A.nonzero()
    prob.linear_constraints.set_coefficients(zip(row_indices.tolist(), col_indices.tolist(), A.data.tolist()))

    # solve the problem
    prob.solve()

    # check the solution status
    print(prob.solution.get_status())
    print(prob.solution.status[prob.solution.get_status()])

    # get the solution
    x_star = prob.solution.get_values()
    obj_star = prob.solution.get_objective_value()

    return(x_star, obj_star)


In [3]:
def quadratic_assignment_problem(frequencies_file, distances_file):
    # your implementation starts below
    freq=np.loadtxt(frequencies_file)
    dist=np.loadtxt(distances_file)
    n=freq.shape[0]
    N=freq.shape[0]*freq.shape[1]

    flist=freq.flatten()
    dlist=dist.flatten()
    cost=np.zeros((N,N))

    for i in range(n):
        for j in range(n):
            for k in range(n):
                for l in range(n):
                    cost[i*n+j,k*n+l]=freq[i,k]*dist[j,l]

    cost=2*cost
    Q = sp.csr_matrix(cost)
    
    c=np.repeat(0,N)
    b=np.repeat(1,2*n)
    senses=np.repeat("E",2*n)
    types=np.repeat("B",N)
    l = np.repeat(0, N)
    u = np.repeat(1, N)

    aij=np.repeat(1,2*N)
    row=np.concatenate((np.repeat(range(n),n),np.repeat(range(n,2*n),n)))
    col=np.concatenate((range(N),np.array(range(N)).reshape(n,n).T.flatten()))
    A = sp.csr_matrix((aij, (row, col)), shape = (2*n,N))
    names=[]
    for i in range(1,n+1):
        for j in range(1,n+1):
            a= "x" + str(i) + str(j)
            names.append(a)
    names=np.array(names)
    x_star, obj_star = mixed_integer_quadratic_programming("Minimize", A, senses, b, c, Q, l, u, types, names)
    X_star=np.array(x_star).reshape(n,n)
    # your implementation ends above
    return(X_star)

In [4]:
X_star = quadratic_assignment_problem("frequencies.txt", "distances.txt")
print(X_star)

Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 41211.000000 after 0.01 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve added 240 rows and 120 columns.
Reduced MIP has 248 rows, 136 columns, and 512 nonzeros.
Reduced MIP has 136 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.04 ticks)
Probing time = 0.02 sec. (0.07 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 168 rows and 48 columns.
Reduced MIP has 80 rows, 88 columns, and 248 nonzeros.
Reduced MIP has 88 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.05 sec. (0.28 ticks)
Probing time = 0.00 sec. (0.04 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 80 rows, 88 columns, and 248 nonzeros.
Reduced MIP has 88 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.18 ticks)
Classifier predicts products in MIQP should be linearized.
Probing time =